# Chapter 8 — Devices, Checkpoints & Debugging (Practice)

Work through these exercises **after reading** `notes/ch08-devices-checkpoints-and-debugging.md`.

Each exercise states the *decision you're practicing*, gives a stub cell to fill in, and is followed by a pre-written **verification cell** — run it to grade yourself. Hand-write your answers in your working copy under `solutions/` (this `template/` copy stays pristine), and don't peek at `solved/` until the verification passes or you're genuinely stuck.

In [ ]:
# ============================================================
# TOPIC: Devices, checkpoint surgery, shape/NaN debugging, reproducibility, memory
# MATH:  softmax denominator = sum(exp(scores)); an all -inf row -> 0/0 -> nan
# REF:   B00 ch08 notes — devices-checkpoints-and-debugging
# ============================================================

# --- Imports ---
import random
import tempfile
import os

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# --- Reproducibility & device ---
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch version : {torch.__version__}")
print(f"device        : {device}  (every exercise here runs fine on CPU)")

CKPT_DIR = tempfile.gettempdir()

## Exercise 1 — The Device-Mismatch Detective

`broken_forward` creates a mask tensor without a `device=` argument, so on a GPU machine it silently defaults to CPU while the model and inputs live on the GPU. Implement `fixed_forward` that creates the mask on the SAME device as its input, and write a `device_checker` you can drop into any function to catch this class of bug early.

**Decision you're practicing:** new tensors default to CPU regardless of where the model lives — the #1 device-mismatch source.

In [ ]:
def broken_forward(embeddings, seq_len):
    """Builds a causal mask WITHOUT matching embeddings' device — the bug."""
    mask = torch.triu(torch.ones(seq_len, seq_len, dtype=torch.bool), diagonal=1)   # defaults to CPU!
    return embeddings, mask

def fixed_forward(embeddings, seq_len):
    """Same mask, but created on embeddings.device."""
    # TODO
    pass

def device_checker(*tensors):
    """Return True if every tensor is on the same device, False otherwise."""
    # TODO
    pass

**Verification**

In [ ]:
# --- Verification: Exercise 1 ---
embeddings = torch.randn(2, 4, 8)
_, fixed_mask = fixed_forward(embeddings, seq_len=4)
assert fixed_mask is not None, "fill in the stub above first"
assert fixed_mask.device == embeddings.device, "fixed_forward's mask must match embeddings' device"
assert fixed_mask.dtype == torch.bool and fixed_mask.shape == (4, 4)

assert device_checker(embeddings, fixed_mask) is True, "matching devices should report True"
same_device_tensor = torch.zeros(3, device=embeddings.device)
assert device_checker(embeddings, same_device_tensor) is True

# a real cross-device mismatch needs a second device; simulate the LOGIC with distinct
# torch.device objects representing "cpu" and a would-be "cuda:0" (no GPU required to test the check)
assert device_checker(torch.zeros(1, device="cpu"), torch.zeros(1, device="cpu")) is True
mismatched_devices = {torch.device("cpu"), torch.device("meta")}   # two genuinely different devices
assert len(mismatched_devices) == 2, "sanity: cpu and meta are different torch.device values"
print("device_checker correctly reports True for matching devices ✓")
print("Exercise 1 passed ✓")

## Exercise 2 — map_location: Simulating a Cross-Machine Load

You can't easily simulate a missing GPU, but you CAN test the mechanics: save a checkpoint, then load it with an explicit `map_location` and confirm every tensor lands where you asked. Implement `save_and_load_with_mapping`.

**Decision you're practicing:** `map_location` overrides a checkpoint's remembered origin device — `"cpu"` is the universally safe choice for a checkpoint of unknown origin.

In [ ]:
def save_and_load_with_mapping(model, path, map_location):
    """Save model.state_dict() to path, then load it back with the given map_location.
    Return the loaded state dict."""
    # TODO
    pass

**Verification**

In [ ]:
# --- Verification: Exercise 2 ---
torch.manual_seed(0)
source_model = nn.Linear(4, 4)
ckpt_path = os.path.join(CKPT_DIR, "b00_ch08_ex2.pt")
loaded = save_and_load_with_mapping(source_model, ckpt_path, map_location="cpu")
assert loaded is not None, "fill in the stub above first"
assert loaded["weight"].device == torch.device("cpu"), "map_location='cpu' must land on cpu"
assert torch.equal(loaded["weight"], source_model.weight), "values must survive the round trip"
assert torch.equal(loaded["bias"], source_model.bias)

# map_location can also be a device object, not just a string
loaded_via_device = save_and_load_with_mapping(source_model, ckpt_path, map_location=torch.device("cpu"))
assert loaded_via_device["weight"].device == torch.device("cpu")
print("checkpoint round-trips correctly with explicit map_location ✓")
print("Exercise 2 passed ✓")

## Exercise 3 — state_dict Surgery: strict=False and Key Renaming

`OldModel` has an `encoder` and a `classifier` head; `NewModel` keeps the same encoder but renames the head to `regressor` (same shapes). Implement `load_with_report` (uses `strict=False`, returns the `_IncompatibleKeys` result) and `load_with_renaming` (fixes the actual mismatch via a dict comprehension, so a **strict** load succeeds).

**Decision you're practicing:** `strict=False` reports what didn't match; renaming keys fixes a known correspondence so you can go back to `strict=True`.

In [ ]:
class OldModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Linear(4, 4)
        self.classifier = nn.Linear(4, 2)

class NewModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Linear(4, 4)
        self.regressor = nn.Linear(4, 2)     # renamed, SAME shape as classifier

def load_with_report(new_model, old_state_dict):
    """Load with strict=False; return the _IncompatibleKeys result."""
    # TODO
    pass

def load_with_renaming(new_model, old_state_dict, rename_map):
    """rename_map: {old_prefix: new_prefix}. Rename matching keys, then load with strict=True."""
    # TODO
    pass

**Verification**

In [ ]:
# --- Verification: Exercise 3 ---
old_model = OldModel()
report = load_with_report(NewModel(), old_model.state_dict())
assert report is not None, "implement load_with_report first"
assert set(report.missing_keys) == {"regressor.weight", "regressor.bias"}, \
    f"missing_keys should be the regressor params, got {report.missing_keys}"
assert set(report.unexpected_keys) == {"classifier.weight", "classifier.bias"}, \
    f"unexpected_keys should be the classifier params, got {report.unexpected_keys}"

fresh_new_model = NewModel()
renamed_result = load_with_renaming(fresh_new_model, old_model.state_dict(), {"classifier": "regressor"})
assert renamed_result.missing_keys == [] and renamed_result.unexpected_keys == [], \
    "after correct renaming, strict=True should succeed with NO incompatible keys"
assert torch.equal(fresh_new_model.encoder.weight, old_model.encoder.weight), "encoder should load unchanged"
assert torch.equal(fresh_new_model.regressor.weight, old_model.classifier.weight), \
    "renamed regressor should carry the OLD classifier's weights"
print("strict=False reported the exact mismatch; renaming fixed it for a strict load ✓")
print("Exercise 3 passed ✓")

## Exercise 4 — Diagnose the Shape-Error Traceback

A pre-written buggy pipeline crashes with a matmul shape error. Read the error message (don't just run it and shrug), identify which of the two shapes is the "impostor," find the line that produced it, and fix that line. Then implement `diagnose_shape_error` — a small helper that extracts the two conflicting shapes from a `RuntimeError`'s message.

**Decision you're practicing:** the detective method from notes §5 — the crash site is not the cause site.

In [ ]:
class BuggyPipeline(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(100, 768)     # produces 768-dim vectors
        self.project = nn.Linear(512, 256)      # ← expects 512-dim input: the impostor number!

    def forward(self, token_ids):
        embedded = self.embed(token_ids)         # (B, T, 768)
        pooled = embedded.mean(dim=1)            # (B, 768)
        return self.project(pooled)              # CRASH: (B, 768) @ (512, 256)^T

buggy = BuggyPipeline()
try:
    buggy(torch.randint(0, 100, (4, 10)))
except RuntimeError as err:
    error_message = str(err)
    print(f"caught: {error_message}")

def diagnose_shape_error(error_message):
    """Extract (shape1, shape2) as tuples of ints from a matmul-shape RuntimeError message."""
    # TODO: parse the two "AxB" patterns out of the message
    pass

# TODO: which module's output shape is the "impostor" relative to the OTHER module's expectation?
impostor_module_name = None   # "embed" or "project"

# TODO: fix BuggyPipeline (in a new class) so forward() runs without crashing
class FixedPipeline(nn.Module):
    def __init__(self):
        super().__init__()
        pass

    def forward(self, token_ids):
        pass

**Verification**

In [ ]:
# --- Verification: Exercise 4 ---
buggy = BuggyPipeline()
try:
    buggy(torch.randint(0, 100, (4, 10)))
    raise AssertionError("BuggyPipeline should have raised a shape error")
except RuntimeError as err:
    message = str(err)

shapes = diagnose_shape_error(message)
assert shapes, "diagnose_shape_error found no shapes — fill in the stub"
flat_numbers = [n for shape in shapes for n in shape]
assert 768 in flat_numbers and 512 in flat_numbers, \
    f"should have found 768 and 512 among the parsed shapes, got {shapes}"

assert impostor_module_name == "project", \
    "project's in_features (512) is what disagrees with what embed actually produces (768)"

fixed = FixedPipeline()
output = fixed(torch.randint(0, 100, (4, 10)))
assert output.shape == (4, 256), f"fixed pipeline should output (4, 256), got {tuple(output.shape)}"
assert fixed.project.in_features == fixed.embed.embedding_dim, \
    "the fix must make project's in_features match embed's embedding_dim"
print(f"parsed the crash shapes {shapes}, identified the impostor, and fixed the pipeline ✓")
print("Exercise 4 passed ✓")

## Exercise 5 — Reproduce and Catch the All-Padding NaN

Reproduce notes §6's exact NaN: a softmax row that is entirely masked to `-inf` (a fully-padded sequence). Implement `finite_loss_check`, a guard that catches a non-finite loss BEFORE it corrupts a training step, and use `torch.autograd.detect_anomaly()` to name the exact backward op that produced the NaN.

**Decision you're practicing:** where NaNs are born (an all-masked softmax row → 0/0) and the two tools for catching them — a cheap `isfinite` guard and the precise (expensive) anomaly detector.

In [ ]:
def attention_with_full_mask_row(scores, fully_masked_row_index):
    """Apply a causal-style mask, then force ONE row to be entirely -inf (fully padded)."""
    masked = scores.clone()
    masked[fully_masked_row_index, :] = float("-inf")
    return torch.softmax(masked, dim=-1)

def finite_loss_check(loss):
    """Return True if loss is finite, False otherwise (does NOT raise)."""
    # TODO
    pass

def find_nan_producing_op(scores, fully_masked_row_index):
    """Run the all-masked-row softmax + backward INSIDE detect_anomaly().
    Return the caught RuntimeError's message, or None if nothing was raised."""
    # TODO
    pass

**Verification**

In [ ]:
# --- Verification: Exercise 5 ---
torch.manual_seed(0)
test_scores = torch.tensor([[1.0, 2.0], [0.5, -0.5]])
probs = attention_with_full_mask_row(test_scores, fully_masked_row_index=1)
assert torch.isnan(probs[1]).all(), "the fully-masked row should be nan (0/0 in softmax)"
assert not torch.isnan(probs[0]).any(), "the untouched row should be perfectly finite"

assert finite_loss_check(probs[0]) is True, "finite_loss_check must return True for a finite tensor"
assert finite_loss_check(probs[1]) is False, "finite_loss_check must return False for the nan row"
assert finite_loss_check is not None

culprit = find_nan_producing_op(test_scores, fully_masked_row_index=1)
assert culprit is not None, "detect_anomaly should have caught something here"
assert "nan" in culprit.lower() or "Softmax" in culprit, f"unexpected anomaly message: {culprit}"
print(f"reproduced the exact NaN, caught it cheaply via isfinite, and named it precisely: {culprit[:80]}...")
print("Exercise 5 passed ✓")

## Exercise 6 — Seed Everything, Including the Part `manual_seed` Misses

`torch.manual_seed` alone does NOT make a pipeline using `random.random()` or `np.random.rand()` reproducible. Implement `set_all_seeds` (per notes §7) and prove it: an unseeded-in-those-two-libraries pipeline gives different results across "runs," while a fully-seeded one is identical.

**Decision you're practicing:** PyTorch's seed doesn't reach Python's `random` or NumPy — each needs its own call.

In [ ]:
def set_all_seeds(seed):
    """Seed random, numpy, and torch (CPU + all CUDA devices, if any)."""
    # TODO
    pass

def mixed_randomness_pipeline():
    """Uses random, numpy, AND torch randomness together -- a realistic data pipeline."""
    python_value = random.random()
    numpy_value = np.random.rand()
    torch_value = torch.rand(1).item()
    return (python_value, numpy_value, torch_value)

**Verification**

In [ ]:
# --- Verification: Exercise 6 ---
torch.manual_seed(999)
run_a_incomplete = mixed_randomness_pipeline()
torch.manual_seed(999)
run_b_incomplete = mixed_randomness_pipeline()
assert run_a_incomplete[1:] != run_b_incomplete[1:] or run_a_incomplete[0] != run_b_incomplete[0], \
    "sanity: torch-only seeding should NOT make random/numpy reproducible"

set_all_seeds(999)
run_a_full = mixed_randomness_pipeline()
set_all_seeds(999)
run_b_full = mixed_randomness_pipeline()
assert run_a_full is not None, "implement set_all_seeds first"
assert run_a_full == run_b_full, f"fully seeded runs must be IDENTICAL: {run_a_full} vs {run_b_full}"

# different seeds must (almost certainly) differ
set_all_seeds(1)
run_seed1 = mixed_randomness_pipeline()
set_all_seeds(2)
run_seed2 = mixed_randomness_pipeline()
assert run_seed1 != run_seed2, "different seeds should give different values"
print(f"torch-only seeding leaves random/numpy unseeded; set_all_seeds makes every run identical ✓")
print("Exercise 6 passed ✓")

## Exercise 7 — Estimate a Training Step's Memory Budget

Implement `estimate_memory_bytes(num_params, dtype_bytes, optimizer_name)`: a back-of-envelope calculator per notes §8 — weights (1×), gradients (1×), and optimizer state (0× for plain SGD, 2× for Adam/AdamW — the `m_t` and `v_t` buffers). Activations are excluded (they depend on batch size and architecture, not just param count).

**Decision you're practicing:** where memory actually goes during training — Adam roughly triples the weights-only footprint before a single activation is counted.

In [ ]:
def estimate_memory_bytes(num_params, dtype_bytes, optimizer_name):
    """Rough (weights + grads + optimizer state) byte estimate. Excludes activations.

    optimizer_name: "sgd" (no extra state) or "adam"/"adamw" (2x extra: m_t, v_t).
    """
    # TODO
    pass

**Verification**

In [ ]:
# --- Verification: Exercise 7 ---
NUM_PARAMS = 10_000_000
FP32_BYTES = 4

sgd_bytes = estimate_memory_bytes(NUM_PARAMS, FP32_BYTES, "sgd")
adamw_bytes = estimate_memory_bytes(NUM_PARAMS, FP32_BYTES, "adamw")
assert sgd_bytes is not None, "fill in the stub above first"

expected_sgd = NUM_PARAMS * FP32_BYTES * 2                    # weights + grads
expected_adamw = NUM_PARAMS * FP32_BYTES * 4                  # weights + grads + 2x optimizer state
assert sgd_bytes == expected_sgd, f"SGD estimate should be {expected_sgd}, got {sgd_bytes}"
assert adamw_bytes == expected_adamw, f"AdamW estimate should be {expected_adamw}, got {adamw_bytes}"
assert abs(adamw_bytes / sgd_bytes - 2.0) < 1e-9, "AdamW should be exactly 2x the SGD estimate here"

# bf16 (2 bytes) should exactly halve every estimate vs fp32 (4 bytes)
bf16_adamw = estimate_memory_bytes(NUM_PARAMS, 2, "adamw")
assert bf16_adamw == adamw_bytes // 2, "halving dtype bytes should exactly halve the estimate"

try:
    estimate_memory_bytes(NUM_PARAMS, FP32_BYTES, "rmsprop")
    raise AssertionError("should have raised on an unknown optimizer name")
except ValueError:
    pass
print(f"SGD: {sgd_bytes/1e6:.1f} MB, AdamW: {adamw_bytes/1e6:.1f} MB (2x SGD), bf16 halves both ✓")
print("Exercise 7 passed ✓")

## Exercise 8 — del and empty_cache Do Not Create Memory

A conceptual exercise with a concrete check: implement `tensors_still_referenced`, which demonstrates that deleting ONE reference to a tensor does not free it while another reference survives — the same principle behind why `del`/`empty_cache()` cannot fix a model that genuinely doesn't fit.

**Decision you're practicing:** freeing memory requires ALL references to be gone, not just the one you typed `del` on — `del` and `empty_cache()` manage bookkeeping, they don't manufacture memory.

In [ ]:
def tensors_still_referenced():
    """Create a tensor, keep a SECOND reference to it, del the first, and prove the
    data is still alive and correct via the second reference. Return the second
    reference's data_ptr() (proof the underlying memory was never released)."""
    # TODO: create big_tensor, alias = big_tensor, del big_tensor, return alias.data_ptr()
    #       (and alias itself, so the caller can inspect the values)
    pass

**Verification**

In [ ]:
# --- Verification: Exercise 8 ---
result = tensors_still_referenced()
assert result is not None, "fill in the stub above first"
original_ptr, alias = result
assert isinstance(alias, torch.Tensor), "must return the surviving tensor reference"
assert alias.numel() == 1000, "the tensor should still have all 1000 elements intact"
assert alias[500].item() == 500.0, "the data must still be correct — nothing was corrupted by del"
assert alias.untyped_storage().data_ptr() == original_ptr, \
    "the underlying storage must be the SAME memory as before del — del only removed a name"
print("del removed a reference, not the memory — a surviving alias kept the data alive and correct ✓")
print("Exercise 8 passed ✓")

---
## Done!

Compare your work against `solved/ch08-devices-checkpoints-and-debugging-solved.ipynb`.

You now have the full survival toolkit: device management, checkpoint surgery, shape-error detective work, NaN hunting, reproducibility, and memory accounting. The **operations block** finishes with **ch09 — Capstone: Text Classifier** — a small, real project assembling every decision guide from chapters 1–8, with one deliberately planted bug for you to find using exactly these tools.